In [1]:
# --- Reproducibility Seed Setup ---
import os
import random
import numpy as np
import torch

SEED = 3180

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print("Global random seed set:", SEED)

Global random seed set: 3180


In [2]:
import sys, site, platform
print("PY:", platform.python_version())
print("EXE:", sys.executable)
print("USER_SITE:", site.getusersitepackages())

PY: 3.9.18
EXE: /common/software/install/manual/pytorch/2.1.2-pyclustertend/bin/python
USER_SITE: /users/4/volko028/.local/lib/python3.9/site-packages


In [3]:
import torch, transformers, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.1.2
transformers: 4.45.2
accelerate: 1.10.1


In [4]:
import os, torch, platform
print("python:", platform.python_version())
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("cuda.is_available():", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count(), "visible:", os.environ.get("CUDA_VISIBLE_DEVICES"))

python: 3.9.18
torch: 2.1.2 cuda: 11.8
cuda.is_available(): True
device_count: 1 visible: 0


In [5]:
# --- Imports ---
from sklearn.model_selection import train_test_split
import math, torch, pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
import numpy as np
from sklearn.metrics import roc_auc_score
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score, roc_auc_score
)
from scipy.special import expit
import numpy as np

2026-04-07 12:39:05.984043: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-07 12:39:07.016263: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-07 12:39:07.016326: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-07 12:39:07.016335: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-07 12:39:07.561732: I tensorflow/core/platform/cpu_feature_g

In [6]:
# --- Config ---
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN    = 256
PER_DEVICE_BS = 2
GRAD_ACCUM    = 4
NUM_EPOCHS    = 3
EVAL_EVERY    = 2000
SAVE_EVERY    = 2000
STRIDE=32

In [7]:
# --- Load data ---
import pandas as pd

use_cols = ["notes", "outcome_critical"]
train_df = pd.read_csv("mv_train_DISPOSITION.csv", usecols = use_cols)
val_df   = pd.read_csv("mv_val_DISPOSITION.csv", usecols = use_cols)
test_df  = pd.read_csv("mv_test_DISPOSITION.csv", usecols = use_cols)

In [8]:
train_df['outcome_critical'].value_counts()

False    135913
True      15973
Name: outcome_critical, dtype: int64

In [9]:
# --- Tokenizer & collator ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

from transformers import DataCollatorWithPadding
import torch

data_collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

def tokenize_with_stride(text: str):
    text = text[:6000]
    return tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        padding=False
    )

In [10]:
# --- Streaming Dataset Loader ---
import math
import pandas as pd
import torch
import random
import numpy as np

class StreamingNotes(torch.utils.data.IterableDataset):
    def __init__(
        self,
        csv_path: str,
        dup_pos: int = 1,
        shuffle_within_chunk: bool = True,
        seed: int = 0,
        tokenizer=None,
        length_cache=None,
        max_len: int = 256,
        stride: int = 32,
        max_char: int = 6000,
        chunksize: int = 8192,
        text_col: str = "notes",
        label_col: str = "outcome_critical",
    ):
        super().__init__()
        self.csv_path = csv_path
        self.dup_pos = max(1, int(dup_pos))
        self.shuffle_within_chunk = shuffle_within_chunk
        self.seed = seed

        self.tokenizer = tokenizer
        self.max_len = int(max_len)
        self.stride = int(stride)
        self.max_char = int(max_char)
        self.chunksize = int(chunksize)

        self.text_col = text_col
        self.label_col = label_col

        self._length = length_cache
        if self._length is None and self.tokenizer is not None:
            self._length = self._estimate_length()

    def __len__(self):
        if self._length is None:
            raise TypeError(
                "StreamingNotes has no length. Pass tokenizer=... so it can estimate __len__, "
                "or pass length_cache=<int>."
            )
        return int(self._length)

    def _tokenize_batch(self, texts):
        return self.tokenizer(
            texts,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            stride=self.stride,
            return_overflowing_tokens=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            padding=False
        )

    def _estimate_length(self):
        total = 0
        usecols = [self.text_col, self.label_col]

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            texts = chunk[self.text_col].astype(str).str.slice(0, self.max_char).tolist()
            labels = chunk[self.label_col].astype(int).to_numpy()

            enc = self._tokenize_batch(texts)
            mapping = enc.get("overflow_to_sample_mapping", None)

            if mapping is None:
                for y in labels:
                    repeats = self.dup_pos if y == 1 else 1
                    total += repeats
            else:
                mapping = np.asarray(mapping)
                for j in range(len(texts)):
                    n_win = int(np.sum(mapping == j))
                    repeats = self.dup_pos if labels[j] == 1 else 1
                    total += repeats * n_win

        return int(total)

    def __iter__(self):
        rng = random.Random(self.seed)
        usecols = [self.text_col, self.label_col]

        global_note_id = 0

        for chunk in pd.read_csv(self.csv_path, chunksize=self.chunksize, usecols=usecols):
            chunk = chunk.dropna(subset=usecols)
            if chunk.empty:
                continue

            rows = list(chunk.iterrows())
            if self.shuffle_within_chunk:
                rng.shuffle(rows)

            for _, row in rows:
                y = int(row[self.label_col])
                text = str(row[self.text_col])[:self.max_char]

                enc = self.tokenizer(
                    text,
                    add_special_tokens=True,
                    truncation=True,
                    max_length=self.max_len,
                    stride=self.stride,
                    return_overflowing_tokens=True,
                    return_attention_mask=True,
                    return_token_type_ids=True,
                    padding=False
                )

                n = len(enc["input_ids"])
                repeats = self.dup_pos if y == 1 else 1

                for _ in range(repeats):
                    for k in range(n):
                        item = {
                            "input_ids": enc["input_ids"][k],
                            "attention_mask": enc["attention_mask"][k],
                            "labels": y,
                            "note_id": global_note_id,
                        }
                        if "token_type_ids" in enc:
                            item["token_type_ids"] = enc["token_type_ids"][k]
                        yield item

                global_note_id += 1
                
                
                
train_ds = StreamingNotes(
    "mv_train_DISPOSITION.csv",
    dup_pos=3,
    shuffle_within_chunk=True,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

val_ds = StreamingNotes(
    "mv_val_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)


In [11]:
# --- Model ---
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: "NoCritical", 1: "Critical"}
model.config.label2id = {"NoCritical": 0, "Critical": 1}

/common/software/install/manual/pytorch/2.1.2-pyclustertend/lib/python3.9/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# --- Training Arguments ---
n_train = len(train_ds)
world_size = max(1, torch.cuda.device_count())
effective_bsz = PER_DEVICE_BS * GRAD_ACCUM * world_size
steps_per_epoch = math.ceil(n_train / effective_bsz)
MAX_STEPS = steps_per_epoch * NUM_EPOCHS
print(f"Training rows={n_train}  steps/epoch={steps_per_epoch}  max_steps={MAX_STEPS}")
assert MAX_STEPS > 0, "No training rows found. Check CSV path and columns."

training_args = TrainingArguments(
    output_dir="runs/clin-longformer",
    seed=SEED,
    data_seed=SEED,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
    eval_accumulation_steps=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=SAVE_EVERY,
    load_best_model_at_end=True,
    metric_for_best_model="note_auroc",
    greater_is_better=True,
    save_total_limit=3,
    remove_unused_columns=False,
    include_inputs_for_metrics=True,
    report_to=["none"],
    gradient_checkpointing=True,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable 

Training rows=184893  steps/epoch=23112  max_steps=69336


TOKENIZERS_PARALLELISM=(true | false)


In [13]:
# --- Create Trainer ---
import numpy as np
import torch
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score

class NoteAwareTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        inputs.pop("note_id", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = dict(inputs)
        note_ids = inputs.pop("note_id", None)
        loss, logits, labels = super().prediction_step(
            model, inputs, prediction_loss_only, ignore_keys
        )

        if note_ids is not None:
            if not isinstance(note_ids, torch.Tensor):
                note_ids = torch.as_tensor(note_ids, dtype=torch.long)
            note_ids = note_ids.view(-1, 1)
            return loss, (logits, note_ids), labels

        return loss, logits, labels

POOL = "mean"

def compute_metrics_note_level(eval_pred):
    preds_obj, labels = eval_pred.predictions, eval_pred.label_ids
    logits, note_ids = preds_obj
    if isinstance(logits, torch.Tensor): logits = logits.detach().cpu().numpy()
    if isinstance(note_ids, torch.Tensor): note_ids = note_ids.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor): labels = labels.detach().cpu().numpy()
    note_ids = note_ids.squeeze(-1)

    logits = np.asarray(logits)
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)
    p1 = probs[:, 1]

    # pool by note
    by_note_probs, by_note_label = {}, {}
    for nid, pr, y in zip(note_ids, p1, labels):
        nid = int(nid)
        by_note_probs.setdefault(nid, []).append(float(pr))
        by_note_label[nid] = int(y)

    if POOL == "max":
        note_probs = np.array([max(v) for nid, v in sorted(by_note_probs.items())])
    else:
        note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
    note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

    # rank metrics
    try:
        auroc = roc_auc_score(note_labels, note_probs)
    except Exception:
        auroc = float("nan")
    try:
        auprc = average_precision_score(note_labels, note_probs)
    except Exception:
        auprc = float("nan")

    # find best F1 threshold on this eval set
    precs, recs, ths = precision_recall_curve(note_labels, note_probs)
    f1s = 2 * precs * recs / (precs + recs + 1e-12)
    best_ix = int(np.nanargmax(f1s))
    best_th = ths[best_ix-1] if best_ix > 0 and best_ix <= len(ths) else 0.5
    target_p = 0.80
    ix_p = np.where(precs >= target_p)[0]

    # report metrics at best_th
    note_pred_best = (note_probs >= best_th).astype(int)
    acc = accuracy_score(note_labels, note_pred_best)
    prec, rec, f1, _ = precision_recall_fscore_support(note_labels, note_pred_best, average="binary", zero_division=0)
    
    print({
      "true_pos_rate": float(np.mean(note_labels)),
      "pred_pos_rate@0.5": float(np.mean((note_probs >= 0.5).astype(int))),
      "mean_prob_pos": float(np.mean(note_probs[note_labels==1])) if np.any(note_labels==1) else None,
      "mean_prob_neg": float(np.mean(note_probs[note_labels==0])) if np.any(note_labels==0) else None,
      "best_th": float(best_th),
    })

    return {
        "note_auroc": auroc,
        "note_auprc": auprc,
        "note_f1": f1,
        "note_precision": prec,
        "note_recall": rec,
        "note_acc": acc,
        "th_best_f1": float(best_th),
    }

In [14]:
# --- Model Training ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from transformers import TrainerCallback
import time

trainer = NoteAwareTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_note_level,
    )
trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss,Note Auroc,Note Auprc,Note F1,Note Precision,Note Recall,Note Acc,Th Best F1
2000,0.525500,0.377623,0.706221,0.273951,0.315489,0.256188,0.410511,0.810755,0.383288
4000,0.570400,0.333690,0.766641,0.377895,0.372881,0.307125,0.474467,0.830454,0.296858
6000,0.592500,0.296859,0.778525,0.395611,0.383937,0.360978,0.410015,0.860213,0.336522
8000,0.510700,0.308835,0.787731,0.411565,0.394531,0.350830,0.450669,0.853050,0.290729
10000,0.576600,0.278475,0.792741,0.426665,0.405656,0.380932,0.433813,0.864953,0.217504
12000,0.558700,0.294113,0.793433,0.428809,0.407476,0.381096,0.437779,0.864742,0.278698
14000,0.678900,0.298949,0.796957,0.426310,0.406982,0.356583,0.473971,0.853260,0.270721
16000,0.462500,0.293070,0.792164,0.429525,0.407487,0.359939,0.469509,0.854946,0.199074
18000,0.543100,0.312130,0.790801,0.426531,0.403002,0.363166,0.452652,0.857527,0.242834
20000,0.492600,0.292690,0.800301,0.437758,0.404217,0.419872,0.389688,0.877963,0.326943


{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.04350574107236911, 'mean_prob_pos': 0.33290371187234435, 'mean_prob_neg': 0.23347413679993745, 'best_th': 0.38328778743743896}
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.07252712525018434, 'mean_prob_pos': 0.39482886173167203, 'mean_prob_neg': 0.1844982388572316, 'best_th': 0.2968580424785614}
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.06678605288107026, 'mean_prob_pos': 0.3491917067970646, 'mean_prob_neg': 0.120356010549768, 'best_th': 0.33652183413505554}
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.05783208680080059, 'mean_prob_pos': 0.38554516101572767, 'mean_prob_neg': 0.14471120487193223, 'best_th': 0.2907290756702423}
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.03813336142420731, 'mean_prob_pos': 0.3198396226554651, 'mean_prob_neg': 0.09519358872799812, 'best_th': 0.21750368177890778}
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.

TrainOutput(global_step=69336, training_loss=0.5253680696726203, metrics={'train_runtime': 22441.8964, 'train_samples_per_second': 24.717, 'train_steps_per_second': 3.09, 'total_flos': 5.794695805484016e+16, 'train_loss': 0.5253680696726203, 'epoch': 3.0000324510259935})

In [15]:
test_ds = StreamingNotes(
    "mv_test_DISPOSITION.csv",
    dup_pos=1,
    shuffle_within_chunk=False,
    seed=SEED,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    stride=STRIDE
)

In [16]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

eval_loader = trainer.get_eval_dataloader(test_ds)

n_val = len(test_ds)
world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_val / bsz))
print(f"n_val={n_val}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)
metrics = compute_metrics_note_level(eval_pred)
print("\nValidation metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))


n_val=19082, bsz=2, total_steps=9541
Predicting: 100%|##########| 9541/9541 [03:17<00:00, 48.34it/s]
{'true_pos_rate': 0.1047087327504477, 'pred_pos_rate@0.5': 0.04482250079005583, 'mean_prob_pos': 0.3096218337146781, 'mean_prob_neg': 0.05864716550969046, 'best_th': 0.13712754845619202}

Validation metrics: {'note_auroc': 0.7995123938122202, 'note_auprc': 0.43314817082150303, 'note_f1': 0.4062797335870599, 'note_precision': 0.38537906137184114, 'note_recall': 0.4295774647887324, 'note_acc': 0.8685347097861582, 'th_best_f1': 0.13712754845619202}

Confusion matrix:
[[15636  1362]
 [ 1134   854]]

Classification report:
              precision    recall  f1-score   support

           0     0.9324    0.9199    0.9261     16998
           1     0.3854    0.4296    0.4063      1988

    accuracy                         0.8685     18986
   macro avg     0.6589    0.6747    0.6662     18986
weighted avg     0.8751    0.8685    0.8717     18986



In [17]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       0         0.013415
1          1       0         0.013274
2          2       0         0.010072
3          3       0         0.038683
4          4       0         0.081593


In [18]:
pred_df

,sample_id,y_true,pred_prob_notes
0,0,0,0.013415
1,1,0,0.013274
2,2,0,0.010072
3,3,0,0.038683
4,4,0,0.081593
...,...,...,...
18981,18981,0,0.432893
18982,18982,0,0.002975
18983,18983,0,0.010449
18984,18984,0,0.017935


In [19]:
from sklearn.metrics import roc_auc_score

auroc = roc_auc_score(note_labels, note_probs)
print("Note-level AUROC:", auroc)

Note-level AUROC: 0.7995123938122202


In [20]:
pred_df.to_csv("notes_test_predictions_task4.csv", index=False)

In [21]:
import sys
import math
import torch
import numpy as np
from tqdm import tqdm
from transformers import EvalPrediction
from sklearn.metrics import confusion_matrix, classification_report

model = trainer.model
model.eval()

# use validation dataset
eval_loader = trainer.get_eval_dataloader(val_ds)

# number of validation examples
n_test = len(val_ds)

world_size = max(1, torch.cuda.device_count())
bsz = trainer.args.per_device_eval_batch_size * world_size

total_steps = max(1, math.ceil(n_test / bsz))
print(f"n_test={n_test}, bsz={bsz}, total_steps={total_steps}", flush=True)

all_logits = []
all_labels = []
all_note_ids = []

eval_iter = iter(eval_loader)

for _ in tqdm(
    range(total_steps),
    desc="Predicting",
    file=sys.stdout,
    disable=False,
    ascii=True,
    mininterval=1.0,
):
    try:
        batch = next(eval_iter)
    except StopIteration:
        break

    batch = {k: v.to(trainer.args.device) for k, v in batch.items()}
    note_ids = batch.pop("note_id")
    labels   = batch["labels"]

    with torch.no_grad():
        outputs = model(**batch)
        logits = outputs.logits

    all_logits.append(logits.cpu().numpy())
    all_labels.append(labels.cpu().numpy())
    all_note_ids.append(note_ids.cpu().numpy())

# concatenate predictions
logits = np.concatenate(all_logits, axis=0)
labels = np.concatenate(all_labels, axis=0)
note_ids = np.concatenate(all_note_ids, axis=0).reshape(-1, 1)

eval_pred = EvalPrediction(
    predictions=(logits, note_ids),
    label_ids=labels,
)

metrics = compute_metrics_note_level(eval_pred)
print("\nTest metrics:", metrics)

# rebuild pooled note probabilities
logits_np = logits
probs = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
p1 = probs[:, 1]

by_note_probs, by_note_label = {}, {}
for nid, pr, y in zip(note_ids.squeeze(-1), p1, labels):
    nid = int(nid)
    by_note_probs.setdefault(nid, []).append(float(pr))
    by_note_label[nid] = int(y)

note_probs = np.array([np.mean(v) for nid, v in sorted(by_note_probs.items())])
note_labels = np.array([by_note_label[nid] for nid in sorted(by_note_label.keys())])

best_th = metrics["th_best_f1"]
note_pred = (note_probs >= best_th).astype(int)

print("\nConfusion matrix:")
print(confusion_matrix(note_labels, note_pred))

print("\nClassification report:")
print(classification_report(note_labels, note_pred, digits=4))

n_test=19091, bsz=2, total_steps=9546
Predicting: 100%|##########| 9546/9546 [03:17<00:00, 48.39it/s]
{'true_pos_rate': 0.10623617402296429, 'pred_pos_rate@0.5': 0.04766670178025914, 'mean_prob_pos': 0.31980764939447187, 'mean_prob_neg': 0.05801426184900042, 'best_th': 0.12847046554088593}

Test metrics: {'note_auroc': 0.8064499079411425, 'note_auprc': 0.4510611329508522, 'note_f1': 0.41764300482425915, 'note_precision': 0.3891267123287671, 'note_recall': 0.45066931085770945, 'note_acc': 0.866480564626567, 'th_best_f1': 0.12847046554088593}

Confusion matrix:
[[15542  1427]
 [ 1108   909]]

Classification report:
              precision    recall  f1-score   support

           0     0.9335    0.9159    0.9246     16969
           1     0.3891    0.4507    0.4176      2017

    accuracy                         0.8665     18986
   macro avg     0.6613    0.6833    0.6711     18986
weighted avg     0.8756    0.8665    0.8707     18986



In [22]:
import pandas as pd

note_ids_sorted = np.array(sorted(by_note_label.keys()))

pred_df = pd.DataFrame({
    "sample_id": note_ids_sorted,
    "y_true": note_labels,
    "pred_prob_notes": note_probs
})

print(pred_df.head())

   sample_id  y_true  pred_prob_notes
0          0       0         0.021105
1          1       0         0.031085
2          2       0         0.023220
3          3       0         0.034456
4          4       1         0.542742


In [23]:
pred_df.to_csv("notes_val_predictions_task4.csv", index=False)